In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..","DTENV"))
Pkg.instantiate()
if size(LOAD_PATH,1) < 4
    push!(LOAD_PATH, joinpath(@__DIR__,"..","scripts"))
end

  Activating project at `~/Blk/JuliaDTFE/DTENV`


4-element Vector{String}:
 "@"
 "@v#.#"
 "@stdlib"
 "/Users/users/spirov/Blk/JuliaDTFE/notebooks/../scripts"

In [2]:
using TetGen
using StaticArrays
using JLD
using BenchmarkTools
using LinearAlgebra
using Plots

In [3]:
include("../scripts/TesselationCore.jl")
import .TesselationCore

BVH = TesselationCore.BVH
point3 = TesselationCore.point3

SVector{3, Float64} (alias for SArray{Tuple{3}, Float64, 1, 3})

In [4]:
import illustris_julia as il


basePath = "../../ThesisMaster/Illustris/";

fields = ["SubhaloMass","SubhaloCM"];
subhalos = il.groupcat.loadSubhalos(basePath,135,fields)

positions = subhalos["SubhaloCM"]
print("done") #otherwise floods GitHub with output 

done

In [20]:
gap = 500
points = positions[:,1:gap:end]
ps = [point3(points[1,i], points[2,i], points[3,i]) for i in 1:size(points,2)]

bvh,tes,tets = TesselationCore.standardEstimator(ps,10)
print("done")

done

In [6]:
N = 1000

width = 75000

step = width/N


xs = bvh.bbox[1,1]:step:bvh.bbox[1,2]
ys = bvh.bbox[2,1]:step:bvh.bbox[2,2]

z = (bvh.bbox[3,2] + bvh.bbox[3,1])/2

dens = zeros(N,N)
print("Done")

Done

In [11]:
pts = [[xs[i],ys[i],z] for i in [50,100,200,500,800]]


ppts = tes.points[tets]


ids = [TesselationCore.findID(pt, ppts, bvh) for pt in pts]


5-element Vector{Union{Nothing, Int64}}:
     nothing
  725
  620
 1152
 1122

In [12]:
valid = findall(!isnothing, ids)


4-element Vector{Int64}:
 2
 3
 4
 5

In [18]:
cleanIds

4-element Vector{Int64}:
  725
  620
 1152
 1122

In [27]:
cleanPoints = points[:,valid]
cleanIds = getindex.(ids[valid]) 

coords = tes.points
rhoStar = tes.ρStar
tets1 = tets[cleanIds, :]


4×4 Matrix{Int32}:
 206   51  12   17
  59   51  12  149
 115  143  43  122
 220  154  85  102

In [28]:
cleanPoints

3×4 Matrix{Float32}:
  1384.44  19501.1  68532.6  73280.2
 26214.2   48023.3  59477.8  17783.0
 17838.1   49247.8  53162.1  23862.6

In [32]:
cuPoints = [CuArray(SVector{3,Float64}(cleanPoint)) for cleanPoint in eachcol(cleanPoints)] 

ErrorException: CUDA driver not functional

In [9]:
TesselationCore.DTFE(pts,bvh,tets,tes)

DimensionMismatch: DimensionMismatch: expected input array of length 3, got length 1

In [11]:
for (i,x) in pairs(xs)
    for (j,y) in pairs(ys)
        dens[i,j] = DTFE([x,y,z],bvh,tets,tes)

    end
end

UndefVarError: UndefVarError: `xs` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [12]:
med = median(dens)

UndefVarError: UndefVarError: `dens` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [13]:
Plots.heatmap(dens ./med,clim=(0,25))

UndefVarError: UndefVarError: `dens` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [12]:
N = 100

width = 75000

step = width/N


xs = bvh.bbox[1,1]:step:bvh.bbox[1,2]
ys = bvh.bbox[2,1]:step:bvh.bbox[2,2]

zs =  bvh.bbox[3,1]:step:bvh.bbox[3,2]

dens = zeros(N,N,N)
print("Done")

Done

In [13]:
DTFE([35000.,35000.,35000.],bvh,tets,tes)

2.1390745266731447e-12

In [14]:
for (i,x) in pairs(xs)
    for (j,y) in pairs(ys)
        for (k,z) in pairs(zs)
            dens[i,j,k] = DTFE([x,y,z],bvh,tets,tes)

        end

    end
end

In [15]:
using ColorSchemes
using GLMakie
lowColor  = get(ColorSchemes.acton,LinRange(0,1,256))[1]

fig = GLMakie.Figure(size = (1600,1600),backgroundcolor=lowColor)
ax = GLMakie.LScene(fig[1,1],scenekw=(show_axis=false,backgroundcolor=lowColor))
volplot = volume!(
    ax,dens ./median(dens),
    algorithm=:mip,
    colormap = :acton,
    colorrange = (.0,25),
    )

Volume{Tuple{Makie.EndPoints{Float32}, Makie.EndPoints{Float32}, Makie.EndPoints{Float32}, Array{Float32, 3}}}

In [16]:
fig

In [19]:
save("../Images/DTFE.png", fig)  